# SPROUT OpenAI embedding — intrinsic dimensionality (reviewer response)

Does SPROUT's OpenAI `text-embedding-3-small` representation (1536-dim) lie on a low-dimensional manifold? Three complementary analyses:

1. Linear PCA scree + cumulative variance
2. Kernel PCA (RBF) eigenvalue spectrum
3. Isomap reconstruction error vs latent dim

Inputs: `data/sprout/sprout_data_train_test.npy` written by `carrot/gen_sprout.py`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import Isomap

os.makedirs('../plots', exist_ok=True)
RNG = np.random.default_rng(42)

In [ ]:
data = np.load('../data/sprout/sprout_data_train_test.npy', allow_pickle=True).item()
XOAI_train = np.asarray(data['XOAI_train']).astype(np.float64)
XOAI_test = np.asarray(data['XOAI_test']).astype(np.float64)
X = np.vstack([XOAI_train, XOAI_test])
print(f'Pooled OpenAI embeddings: shape={X.shape}  ||x||_2 mean/std = {np.linalg.norm(X, axis=1).mean():.3f} / {np.linalg.norm(X, axis=1).std():.3f}')

# Center (OpenAI embeddings are L2-normalized but not mean-zero)
X_c = StandardScaler(with_mean=True, with_std=False).fit_transform(X)

## 1. Linear PCA

In [ ]:
pca = PCA().fit(X_c)
eigvals = pca.explained_variance_          # raw eigenvalues of the centered covariance
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)

thresholds = [0.9, 0.95, 0.99]
dims_needed = {t: int(np.searchsorted(cum, t) + 1) for t in thresholds}
for t, d in dims_needed.items():
    print(f'  {int(t*100)}% cumulative variance: {d} components')

# Classical Cattell scree: raw eigenvalues on a linear y-axis. Zoomed to top 100
# so the elbow is readable; full 1536-component curve is heavy-tailed and would
# compress everything after ~PC5 to the axis.
TOP = 100
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(range(1, TOP + 1), eigvals[:TOP], marker='o', ms=2.5, lw=1)
axes[0].set_xlabel('component index')
axes[0].set_ylabel('eigenvalue')
axes[0].set_title(f'Scree plot (top {TOP})')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(cum) + 1), cum, lw=1)
for t, d in dims_needed.items():
    axes[1].axhline(t, color='grey', linestyle=':', lw=0.8)
    axes[1].axvline(d, color='grey', linestyle=':', lw=0.8)
    axes[1].annotate(f'{int(t*100)}% @ {d}', xy=(d, t), xytext=(d + 20, t - 0.04), fontsize=8)
axes[1].set_xlabel('component index')
axes[1].set_ylabel('cumulative variance')
axes[1].set_title('Cumulative variance')
axes[1].set_ylim(0, 1.02)
axes[1].grid(True, alpha=0.3)

fig.suptitle(f'SPROUT OpenAI embedding — PCA  (n={X.shape[0]}, d=1536)')
fig.tight_layout()
fig.savefig('../plots/sprout_pca_scree.pdf', bbox_inches='tight')
plt.show()

## 2. Kernel PCA (RBF) — nonlinear scree

In [ ]:
n_sub_kpca = min(5000, X_c.shape[0])
sub_idx = RNG.choice(X_c.shape[0], size=n_sub_kpca, replace=False)
X_sub = X_c[sub_idx]
print(f'KernelPCA on {n_sub_kpca} points')

gamma = 1.0 / X.shape[1]
kpca = KernelPCA(kernel='rbf', n_components=50, gamma=gamma, random_state=42).fit(X_sub)
eigvals_k = getattr(kpca, 'eigenvalues_', getattr(kpca, 'lambdas_', None))
eigvals_k = np.sort(np.asarray(eigvals_k))[::-1]

# Classical scree: raw eigenvalues on linear y. Range here is ~10x so a single
# linear panel shows the elbow well without needing log scale.
fig, ax = plt.subplots(1, 1, figsize=(4.5, 3.5))
ax.plot(range(1, len(eigvals_k) + 1), eigvals_k, marker='o', ms=3, lw=1)
ax.set_xlabel('component index')
ax.set_ylabel('eigenvalue')
ax.set_title(f'RBF Kernel PCA scree  (γ=1/d, n={n_sub_kpca})')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('../plots/sprout_kpca_scree.pdf', bbox_inches='tight')
plt.show()

# Print where the eigenvalue 'knee' is by fractional drop
drops = eigvals_k[:-1] / eigvals_k[1:]
print('top-10 eigenvalue ratios (λ_k / λ_{k+1}):', np.round(drops[:10], 2))

## 3. Isomap reconstruction error vs latent dim

In [ ]:
n_sub_iso = min(3000, X_c.shape[0])
sub_idx = RNG.choice(X_c.shape[0], size=n_sub_iso, replace=False)
X_sub = X_c[sub_idx]
print(f'Isomap on {n_sub_iso} points')

ks = [2, 5, 10, 20, 50]
recon_errs = []
for k in ks:
    iso = Isomap(n_components=k, n_neighbors=15).fit(X_sub)
    err = float(iso.reconstruction_error())
    recon_errs.append(err)
    print(f'  k={k:>3}  reconstruction_error = {err:.4f}')

fig, ax = plt.subplots(1, 1, figsize=(4.5, 3.5))
ax.plot(ks, recon_errs, marker='o')
ax.set_xlabel('Isomap latent dim k')
ax.set_ylabel('reconstruction error')
ax.set_title(f'Isomap reconstruction error  (n={n_sub_iso})')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('../plots/sprout_isomap_recon.pdf', bbox_inches='tight')
plt.show()

## Summary (auto-populate into rebuttal text)

- PCA: reported in the cell above — cumulative variance thresholds at 90%, 95%, 99%.
- Kernel PCA (RBF, γ=1/d): the spectrum above shows how quickly nonlinear variance is captured; look for the knee in the log-scale plot.
- Isomap: the elbow in the reconstruction-error curve is an intrinsic-dim estimate.